## Copy into and merge commands

COPY INTO Command
- INcrementlay loads data into Delta lake tables from cloud storage
- Supports schema evolution
- Support wide range of file formats (csv, json, parquet, delta)
- Alternative to auto loader for batch ingestion


[COPY into doc ](https://docs.databricks.com/aws/en/sql/language-manual/delta-copy-into)

In [0]:
-- Create the table to copy the data into

create table if not exists demo.delta_lake.raw_stock_prices;


In [0]:
-- incremently load new files into the table
delete from demo.delta_lake.raw_stock_prices;

copy into demo.delta_lake.raw_stock_prices
from 'abfss://demo@deacourseexternaldl.dfs.core.windows.net/landing/stock_prices'
fileformat = JSON
format_options ('inferSchema' = 'true')
copy_options ('mergeSchema' = 'true');

In [0]:
select * from demo.delta_lake.raw_stock_prices

## MERGE statement
- Used for upsert (insert/ update/ delete operations in a single statement)
- allows merging new data into a target table based on matching conditon

[MERGE into doc](https://docs.databricks.com/aws/en/sql/language-manual/delta-merge-into)

In [0]:
-- create the table to merge the data into

create table if not exists demo.delta_lake.stock_prices 
(
    stock_id string,
    price double,
    trading_date date
) 

## Merge the source data into target table
- Insert new stocks received
- Update price and trading_date if updates received
- Delete stocks which are de-listed from the exchange (status = 'DELISTED')

In [0]:
merge into demo.delta_lake.stock_prices as target
using demo.delta_lake.raw_stock_prices as source
    on target.stock_id = source.stock_id
when matched and source.status = 'ACTIVE' then
    update set target.price = source.price, target.trading_date = source.trading_date
when matched and source.status = "DELISTED" then
    delete
when not matched and source.status = "ACTIVE" then
    insert (stock_id, price, trading_date) values (source.stock_id, source.price, source.trading_date)

In [0]:
select * from demo.delta_lake.stock_prices